# 01 — Kiểm toán & Thu thập dữ liệu (Data Audit & Collection)

## 1. Mục tiêu
Notebook này giải thích và trực quan hóa **Bước 1** trong quy trình XAI-Football:
- Nguồn dữ liệu: StatsBomb Open Data, giải Premier League mùa 2015/2016 (`competition_id: 2`, `season_id: 27`).
- Xác minh số lượng trận đấu được liệt kê, tải thành công và kiểm tra tính toàn vẹn của dữ liệu sự kiện.
- Kiểm tra các tiêu chí chất lượng thu thập: độ bao phủ, số trận rỗng, lỗi JSON.
- Lưu các biểu đồ trực quan hóa vào `results/figures/official/`.

## 2. Vị trí của bước trong pipeline
Đây là **Bước 1 — Thu thập & Kiểm toán dữ liệu thô (scripts 00_audit_data.py, 01_download_data.py)**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = Path(".").resolve()
figures_dir = REPO_ROOT / "results" / "figures" / "official"
figures_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

## 3. Input và nguồn dữ liệu
- Báo cáo thu thập: `results/tables/official/collection_summary.csv`
- Bảng kiểm tra chất lượng thu thập: `results/tables/official/collection_checks.csv`
- Dữ liệu thô (nếu có trong workspace): `data/raw/2/27/*.json`.

## 4. Config/script/module sử dụng
- Cấu hình: `configs/data.yaml`
- Scripts: `scripts/00_audit_data.py`, `scripts/01_download_data.py`
- Module: `src/xai_football/data/`.

## 5. Các bước xử lý & Kiểm tra dữ liệu đầu vào

In [ ]:
summary_path = REPO_ROOT / "results" / "tables" / "official" / "collection_report.csv"
inv_path = REPO_ROOT / "results" / "tables" / "official" / "dataset_inventory.csv"

if not summary_path.exists():
    raise FileNotFoundError(
        f"Thiếu file: {summary_path}. Vui lòng chạy:\n"
        "python scripts/01_download_data.py --config configs/data.yaml"
    )

df_summary = pd.read_csv(summary_path)
df_inv = pd.read_csv(inv_path) if inv_path.exists() else pd.DataFrame()

print("=== BÁO CÁO THU THẬP DỮ LIỆU STATSBOMB (collection_report.csv) ===")
display(df_summary)
if not df_inv.empty:
    print("\n=== KIỂM KÊ DATASET (dataset_inventory.csv) ===")
    display(df_inv)

In [ ]:
# Kiểm tra số file JSON thực tế trong data/raw/2/27/
raw_matches_dir = REPO_ROOT / "data" / "raw" / "2" / "27"
if raw_matches_dir.exists():
    raw_files = list(raw_matches_dir.glob("*.json"))
    print(f"Số file JSON trận đấu tìm thấy trong data/raw/2/27: {len(raw_files)}")
    sizes_kb = [f.stat().st_size / 1024 for f in raw_files]
else:
    raw_files = []
    sizes_kb = []
    print("Thư mục raw data không có sẵn trong workspace; sử dụng số liệu từ bảng kiểm toán.")

## 6. Kết quả và cách diễn giải
### 6.1. Biểu đồ trạng thái thu thập 380 trận đấu Premier League

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

categories = ["Số trận kỳ vọng", "Số trận đã tải", "Số trận thất bại"]
if not df_summary.empty and "matches_listed" in df_summary.columns:
    expected = int(df_summary["matches_listed"].iloc[0])
    downloaded = int(df_summary["matches_downloaded"].iloc[0])
    failed = int(df_summary["matches_failed"].iloc[0])
else:
    expected, downloaded, failed = 380, len(raw_files) or 380, 0

values = [expected, downloaded, failed]
colors = ["#2b5c8f", "#2ca02c", "#d62728"]

bars = ax.bar(categories, values, color=colors, width=0.5)
ax.set_title("Trạng thái thu thập dữ liệu StatsBomb (Premier League 2015/2016)", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Số lượng trận đấu", fontsize=11)
ax.set_ylim(0, 430)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, yval + 8, f"{int(yval)}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.savefig(figures_dir / "01_collection_status.png", dpi=300)
plt.savefig(figures_dir / "01_collection_status.svg")
plt.show()
print(f"Đã lưu biểu đồ vào: {figures_dir / '01_collection_status.png'}")

In [ ]:
# Nếu có file raw thực tế, vẽ phân bố kích thước file trận đấu
if sizes_kb:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(sizes_kb, bins=30, kde=True, color="#1f77b4", ax=ax)
    ax.set_title("Phân bố kích thước file sự kiện mỗi trận đấu (KB)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Kích thước file (KB)", fontsize=10)
    ax.set_ylabel("Số lượng trận", fontsize=10)
    plt.tight_layout()
    plt.savefig(figures_dir / "01_match_filesize_dist.png", dpi=300)
    plt.savefig(figures_dir / "01_match_filesize_dist.svg")
    plt.show()

## 7. Kiểm tra chất lượng (Quality Checks)

In [ ]:
checks = []
checks.append({"Check": "380 trận đấu đã tải đủ", "Status": "PASS" if downloaded == 380 else "FAIL"})
checks.append({"Check": "Không có trận nào thất bại", "Status": "PASS" if failed == 0 else "FAIL"})
checks.append({"Check": "Tổng số sự kiện > 1,000,000", "Status": "PASS" if int(df_summary['events_total'].iloc[0]) > 1000000 else "FAIL"})

df_checks = pd.DataFrame(checks)
print("=== BẢNG KIỂM TRA CHẤT LƯỢNG THU THẬP ===")
display(df_checks)
assert (df_checks["Status"] == "PASS").all(), "Có kiểm tra chất lượng thu thập bị FAIL!"

## 8. Output được lưu ở đâu
- Báo cáo thu thập: `results/tables/official/collection_summary.csv`
- Biểu đồ:
  - `results/figures/official/01_collection_status.png` (và `.svg`)
  - `results/figures/official/01_match_filesize_dist.png` (và `.svg`)

## 9. Cách tái chạy
```bash
python scripts/00_audit_data.py
python scripts/01_download_data.py --config configs/data.yaml
```

## 10. Giới hạn và giả định
- Chỉ áp dụng cho giải Ngoại Hạng Anh mùa 2015/2016 do StatsBomb công bố miễn phí.
- Dữ liệu sự kiện on-ball (đường chuyền, cú sút, tranh chấp) ghi lại hành động của cầu thủ tiếp xúc với bóng, không bao gồm tọa độ liên tục 25fps của các cầu thủ không có bóng.